# 🔬 Multimodal Gemini-based Document Forgery Detection Practice
In this practice, we will build a high-precision forensic pipeline to detect digital manipulation traces in documents such as receipts, invoices, and transaction statements using the latest Google GenAI SDK and Gemini models.

### 🎯 Learning Objectives
1. Set up the latest `google-genai` SDK environment and integrate with Vertex AI (IAM) / AI Studio clients.
2. Build an interactive image upload environment for multimodal analysis.
3. Leverage Structured Outputs constraints for stable forensic data extraction.

In [ ]:
# Install libraries (Latest official Google GenAI SDK and related tools)
!pip install -q google-genai pandas pillow matplotlib openpyxl
print("✅ Essential libraries installation completed.")

## 🔑 STEP 2. Initializing Google Gemini Client
Google's latest GenAI SDK provides a unified interface.
* **AI Studio Mode**: Works when you enter your key in `API_KEY`.
* **Vertex AI IAM Mode**: Automatically recognizes infrastructure account permissions (IAM) and integrates when you enter `PROJECT_ID` and `LOCATION`.

In [ ]:
import os
import getpass
from google import genai
from google.genai import types

# 1. Remove previous invalid object reference and initialize variables
if 'client' in globals():
    del client

# 2. Authentication setup (API_KEY must be left empty when using Vertex AI IAM mode)
API_KEY = ""
PROJECT_ID = "claude-code-dev-492600"
LOCATION = "global"

def get_client():
    # If API Key explicitly exists, use AI Studio mode
    if API_KEY.strip():
        print("🔑 Initializing AI Studio Client...")
        return genai.Client(api_key=API_KEY)

    # If Project ID exists, use Vertex AI IAM mode (API_KEY omitted)
    if PROJECT_ID.strip():
        print(f"🌐 Initializing Vertex AI Client (IAM) for project '{PROJECT_ID}'...")
        return genai.Client(vertexai=True, project=PROJECT_ID, location=LOCATION)

    # If neither is set, prompt for manual input
    key = getpass.getpass("Enter your GEMINI_API_KEY: ")
    return genai.Client(api_key=key)

try:
    client = get_client()
    print("✅ Client initialized successfully.")
except Exception as e:
    print(f"❌ Initialization Error: {e}")

## 📤 STEP 3. Loading and Visualizing Target Images for Analysis
In a Google Colab environment, you can upload local files through a browser popup. In a local Jupyter environment, it automatically searches the working directory or the '허위서류 공유' (Shared Forged Documents) folder for images.

In [ ]:
import glob
from PIL import Image
import matplotlib.pyplot as plt

image_paths = []

# 1. Interactive file upload support in Google Colab / Colab Enterprise environment
try:
    from google.colab import files
    print("📤 Please upload the forged document image files to practice with (Multiple selection possible)...")
    uploaded = files.upload()
    for filename in uploaded.keys():
        image_paths.append(filename)
    print(f"✅ Uploaded files list: {image_paths}")
except ImportError:
    # 2. Local development environment fallback: Automatic search in '허위서류 공유' folder or current directory
    print("💻 Local execution environment detected. Searching for images in the '허위서류 공유' folder or the current working directory...")
    image_folder = "허위서류 공유"
    if os.path.exists(image_folder):
        found_files = glob.glob(f"{image_folder}/*.*")
        image_paths = sorted([p for p in found_files if p.lower().endswith(('.png', '.jpg', '.jpeg'))])
    else:
        found_files = glob.glob("*.*")
        image_paths = sorted([p for p in found_files if p.lower().endswith(('.png', '.jpg', '.jpeg'))])

if not image_paths:
    print("⚠️ No image files found or uploaded. To proceed, please upload image files or place them in the '허위서류 공유' folder.")
else:
    print(f"📂 Total of {len(image_paths)} image file(s) loaded:")
    for path in image_paths:
        print(f"  - {os.path.basename(path)}")

# Helper function for image grid visualization
def show_images_grid(paths):
    if not paths:
        return
    fig, axes = plt.subplots(1, len(paths), figsize=(20, 10))
    if len(paths) == 1:
        axes = [axes]
    for ax, p in zip(axes, paths):
        try:
            img = Image.open(p)
            ax.imshow(img)
            ax.set_title(os.path.basename(p), fontsize=10)
            ax.axis("off")
        except Exception as e:
            print(f"❌ Failed to open {p}: {e}")
    plt.tight_layout()
    plt.show()

if image_paths:
    show_images_grid(image_paths)

## 🧠 STEP 4. Defining Forensic System Prompt and Analysis Pipeline
We define an **English system prompt** containing global standards for forgery analysis. The LLM's reasoning and logical development are executed precisely in English, and using the `response_schema` rules, the final output fields are structured cleanly.

### 🔍 Key Forensic Checkpoints
1. **Logical Inconsistencies**: Analysis of temporal/formal causal relationships between payment date, approval number, and barcode.
2. **Visual Alterations**: Fine pixelation (compression noise) around specific characters, mismatched font spacing/thickness.
3. **AI Generation**: Capture traces after browser text alteration (Inspect Element), blurred or corrupted fake logos.

In [ ]:
import json

# =====================================================================
# 1. High-Detail Forensic Document Analysis System Prompt (English)
# =====================================================================
FORENSIC_SYSTEM_PROMPT = """
You are an expert Forensic Document Examiner and Fraud Prevention Analyst specializing in documents such as receipts, invoices, tax statements, and official transaction screenshots.
Your sole mission is to analyze the provided document image to identify if it is faked, forged, altered, or digitally manipulated, and then output your final expert judgment into three specific fields.

### DETAILED INSPECTION PROTOCOLS
You must meticulously scan the document image for the following indicators of fraud:

1. Textual, Grammatical, and Logical Inconsistencies:
   - Date & Time Logic: Verify if the payment dates, approval dates, or order dates logically align with surrounding text, order numbers, approval codes, or any visible URL timestamps.
   - Typographical & Brand Frauds: Look for spelling mistakes, unnatural phrasing, or subtle typosquatting in famous store/brand names (especially checking for character level alterations in non-English texts like Korean).
   - Address Cross-Referencing: Check if the branch office name matches the format of the provided road-name or lot-number address.
   - URL/Domain Anomaly: Detect suspicious domains, phishing paths, or altered web address text in screenshots.

2. Visual and Digital Tampering Indicators:
   - Font & Border Discrepancies: Scan for uneven text alignments, inconsistent font weights, mixed typefaces within the same line, or unnatural pixel artifacts/compression noise around text borders.
   - Digital Composition & Cut-and-Paste: Look for differences in background noise, texture mismatches, or straight-line clipping artifacts around critical numbers (amounts, dates, approval numbers).
   - Inspect Element Modification: Check for unnaturally perfect system fonts layered over compressed image backgrounds, which indicates browser text alteration before capturing.

3. AI Layout and Synthetic Template Generation:
   - Synthetic Structural Templates: Identify unnatural spacing, missing dividing lines, or geometrically distorted grids.
   - Hallucinated Graphics: Spot distorted corporate logos, deformed barcodes, corrupted QR codes, or unreadable miniature text elements that are typical in AI-generated images.

### OUTPUT JSON REQUIREMENT
You must output a single, valid JSON object matching the requested schema exactly. 
All string explanations in the 'reason' field MUST be written in English, provided with specific forensic context.
"""

# =====================================================================
# 2. Execution & API Call Function (Structured Outputs)
# =====================================================================
def analyze_document_forgery(image_path: str, model_name: str = "gemini-3.5-flash") -> dict:
    if not os.path.exists(image_path):
        return {"error": f"File not found: {image_path}"}
        
    try:
        img = Image.open(image_path)
        
        response = client.models.generate_content(
            model=model_name,
            contents=[img],
            config=types.GenerateContentConfig(
                system_instruction=FORENSIC_SYSTEM_PROMPT,
                response_mime_type="application/json",
                # Hard binding of JSON schema constraints in config without Pydantic declaration
                response_schema={
                    "type": "OBJECT",
                    "properties": {
                        "is_forged": {
                            "type": "BOOLEAN", 
                            "description": "Final verdict. True if the document has any signs of forgery, alteration, or digital tampering. Otherwise, False."
                        },
                        "confidence_score": {
                            "type": "NUMBER", 
                            "description": "Confidence level of the verdict, ranging from 0.0 (no confidence) to 1.0 (absolute certainty)."
                        },
                        "reason": {
                            "type": "STRING", 
                            "description": "Detailed forensic reasoning and listed anomalies that led to this verdict. MUST BE WRITTEN IN ENGLISH."
                        }
                    },
                    "required": ["is_forged", "confidence_score", "reason"]
                },
                temperature=0.1  # Set lowest temperature for forensic precision and deterministic results
            )
        )
        # Instantly parse returned raw JSON string into a Python dictionary
        return json.loads(response.text)
        
    except Exception as e:
        return {"error": f"Exception occurred during model execution: {str(e)}"}

## 🔬 STEP 5. Running the Pipeline and Checking Forgery Benchmark Results
We iterate over the uploaded documents and perform batch forgery detection analysis. Thanks to the Structured Outputs mode, the JSON format is parsed reliably without formatting issues.

In [ ]:
# =====================================================================
# 3. Multi-Image Evaluation Loop & Console Output (English Presentation)
# =====================================================================
if image_paths:
    print("🔬 Running high-precision document forgery detection pipeline...")
    
    for path in image_paths:
        file_name = os.path.basename(path)
        print(f"\n--------------------------------------------------\n📄 Analyzing File: {file_name}")
        
        # Run analysis algorithm (Default model: gemini-3.5-flash)
        result = analyze_document_forgery(path, "gemini-3.5-flash")
        
        # Exception & Error handling
        if "error" in result:
            print(f"  ❌ {result['error']}")
            continue
            
        # Re-mapping data for clear presentation
        verdict_str = "⚠️ Suspected Forgery/Alteration (FORGED)" if result["is_forged"] else "✅ Genuine Document (GENUINE)"
        confidence_pct = result["confidence_score"] * 100
        
        print(f"  [Verdict] {verdict_str}")
        print(f"  [Confidence] {confidence_pct:.1f}%")
        print(f"  [Detailed Analysis] {result['reason']}")
else:
    print("❌ No images to analyze. Please check STEP 3.")